# TrendPulse — Part 2: NumPy & Pandas Data Wrangling

Part 1 turned the YouTube "Trending" lists into clean Python structures. This notebook uses **NumPy** and **Pandas** to compute a transparent engagement score, clean and inspect the data, and answer concrete questions about *what trends and for how long*.

**Self-contained:** this notebook reads only from this folder's own `data/` directory (`USvideos.csv`, `GBvideos.csv`, `INvideos.csv`, `categories.csv`). It needs **no network access** and does not import any file from Part 1.

| Section | Brief task | What it does |
|---|---|---|
| 0 | Setup | Config, loading, carrying Part 1's cleaning rules over |
| 1 | NumPy arrays | Arrays, `shape`, `dtype`, indexing and slicing |
| 2 | Engagement score | Ratios, normalisation, weighted score, log scale, top-10% mask |
| 3 | Reshape and broadcast | Days × countries array, shares, relative values |
| 4 | Loading and inspection | `head`/`info`/`describe`/`shape`, type fixes, data-quality issues |
| 5 | Filtering | Boolean indexing, `loc`, `iloc`, sorting |
| 6 | Missing values and duplicates | `isna`, `fillna`, `dropna`, `drop_duplicates` |
| 7 | Joining category names | `merge` with `categories.csv` |
| 8 | Aggregation | `groupby`/`agg`, `value_counts`, publishing day and hour |
| 9 | Time to trend and lifespan | `days_to_trend`, `nunique` lifespan |
| 10 | Combining data | `concat`, per-video and per-channel `merge` |

## 0. Setup and data loading

### How this notebook relates to Part 1
Part 2 must stand on its own, so it does **not** read Part 1's `cleaned_records`. Instead it carries over the same *cleaning decisions* from Part 1 and applies them with Pandas:

| Part 1 decision | Applied here |
|---|---|
| A duplicate is the same `country` + `video_id` + `trending_date`; keep the first copy | `drop_duplicates(subset=DUP_KEY, keep="first")` |
| `trending_date` is stored as `yy.dd.mm` (e.g. `17.14.11` = 14 Nov 2017) | `DATE_FORMAT = "%y.%d.%m"` |
| Blank `likes` / `comment_count` stay missing — never replaced by `0` | Left as `NaN` |
| Loader that checks required columns and tags each row with its country | `load_country()` below (copied from Part 1 and trimmed) |

**Notebook discipline:** all settings live in the first code cell, raw data is never overwritten, and the notebook is meant to be run top to bottom.

In [112]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# ---------- Files and countries ----------
DATA_DIR = Path("data")
COUNTRIES = ["US", "GB", "IN"]
CATEGORY_FILE = DATA_DIR / "categories.csv"

# ---------- Cleaning rules carried over from Part 1 ----------
DUP_KEY = ["country", "video_id", "trending_date"]   # what counts as a duplicate
DATE_FORMAT = "%y.%d.%m"                             # trending_date is yy.dd.mm
NUMERIC_COLUMNS = ["views", "likes", "dislikes", "comment_count"]
EXCLUDE_REMOVED = True    # drop videos flagged video_error_or_removed from the analysis table (Section 4)

# ---------- Engagement-score weights (justified in Section 2) ----------
W_VIEWS, W_LIKES, W_COMMENTS = 0.2, 0.3, 0.5
TOP_SHARE = 0.10          # flag the top 10% of records by engagement score

REQUIRED_COLUMNS = [
    "video_id", "trending_date", "title", "channel_title", "category_id",
    "publish_time", "views", "likes", "dislikes", "comment_count",
    "comments_disabled", "ratings_disabled", "video_error_or_removed", "description",
]

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.3.3 | numpy 2.3.5


In [113]:
def load_country(country):
    """Read one country's CSV with pd.read_csv() and add a 'country' column."""
    path = DATA_DIR / f"{country}videos.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Copy the country CSV files and categories.csv into '{DATA_DIR}/'."
        )
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")
    df["country"] = country
    return df


# One DataFrame per country, then stack them into one with concat() (used again in Section 10)
country_frames = {country: load_country(country) for country in COUNTRIES}
combined_raw = pd.concat(country_frames.values(), ignore_index=True)

categories = pd.read_csv(CATEGORY_FILE)

for country, frame in country_frames.items():
    print(f"{country}: {len(frame):,} rows loaded")
print(f"Combined (raw): {len(combined_raw):,} rows, {combined_raw.shape[1]} columns")
print(f"Categories    : {len(categories)} rows")

US: 16,080 rows loaded
GB: 15,075 rows loaded
IN: 15,075 rows loaded
Combined (raw): 46,230 rows, 17 columns
Categories    : 17 rows


### Deduplicated working table: `trending`
Every later section starts from `trending`: the combined data after removing duplicate rows with the Part 1 rule. Section 6 reports the duplicates in detail. `reset_index(drop=True)` gives a clean `0…n-1` index, so the NumPy arrays built in Sections 1–2 line up exactly with the DataFrame rows.

In [114]:
trending = combined_raw.drop_duplicates(subset=DUP_KEY, keep="first").reset_index(drop=True)
print(f"Raw rows: {len(combined_raw):,}  ->  after removing duplicates: {len(trending):,}")

Raw rows: 46,230  ->  after removing duplicates: 46,000


## 1. NumPy arrays

The four count columns become NumPy arrays. `views` has no gaps, so it stays an integer array. `likes` and `comment_count` contain blanks, which NumPy can only store as `NaN` (a float), so they are `float64`. `dislikes` is cast to `float64` too so all later arithmetic uses the same type.

In [115]:
views = trending["views"].to_numpy()
likes = trending["likes"].to_numpy(dtype=float)
dislikes = trending["dislikes"].to_numpy(dtype=float)
comment_count = trending["comment_count"].to_numpy(dtype=float)

arrays = {"views": views, "likes": likes, "dislikes": dislikes, "comment_count": comment_count}

print(f"{'array':<15}{'shape':<12}{'dtype':<10}{'NaN count':>10}")
print("-" * 47)
for name, array in arrays.items():
    print(f"{name:<15}{str(array.shape):<12}{str(array.dtype):<10}{np.isnan(array).sum():>10}")

array          shape       dtype      NaN count
-----------------------------------------------
views          (46000,)    int64              0
likes          (46000,)    float64           12
dislikes       (46000,)    float64            0
comment_count  (46000,)    float64           36


### Indexing and slicing

In [116]:
# The 10 largest view counts: sort ascending, reverse with [::-1], keep the first 10
print("Top 10 view counts:")
print(np.sort(views)[::-1][:10])

# Every 100th record: slice with a step of 100
every_100th = views[::100]
print("\nEvery 100th record -> shape", every_100th.shape, "| first 10:", every_100th[:10])

# Sub-array of my choice (1): the first 10 likes
print("\nFirst 10 likes:", likes[:10])

# Sub-array of my choice (2): a 2-D slice. Stack the arrays as columns, take 5 rows x first 2 columns
stats = np.column_stack((views, likes, dislikes, comment_count))
print("\nstats shape:", stats.shape, "| dtype:", stats.dtype)
print("First 5 rows, columns [views, likes]:")
print(stats[:5, :2])

Top 10 view counts:
[360760164 343297002 286714357 282812154 255362869 236275528 230704137
 210747294 200444300 181785943]

Every 100th record -> shape (460,) | first 10: [ 48555  21869 245258  61340 359529  58838 430016 192950 537352 279993]

First 10 likes: [ 2189.  1159.  8742.   176. 11305. 15099.  5347. 52635.  3509.  6958.]

stats shape: (46000, 4) | dtype: float64
First 5 rows, columns [views, likes]:
[[4.85550e+04 2.18900e+03]
 [1.98385e+05 1.15900e+03]
 [2.31656e+05 8.74200e+03]
 [8.08300e+03 1.76000e+02]
 [3.73161e+05 1.13050e+04]]


> Each row is one video *on one trending day*, so a very popular video can appear on several rows. The 10 largest view counts are therefore 10 records, not necessarily 10 different videos.

## 2. Engagement score without loops

Everything in this section uses whole-array operations — no `for` loops.

### (a) Like ratio and comment rate with safe division
- **Like ratio** = `likes / (likes + dislikes)`
- **Comment rate** = `comment_count / views`

`np.divide(..., where=...)` only divides where the denominator is non-zero; everywhere else it keeps the `NaN` placed in `out`. A blank `likes` or `comment_count` also gives `NaN`, because `NaN` stays `NaN` in arithmetic.

In [117]:
votes = likes + dislikes

like_ratio = np.divide(likes, votes, out=np.full_like(likes, np.nan), where=(votes != 0))
comment_rate = np.divide(comment_count, views, out=np.full_like(comment_count, np.nan), where=(views != 0))

print("like_ratio   -> shape", like_ratio.shape, "| NaN:", np.isnan(like_ratio).sum(),
      "| median:", round(float(np.nanmedian(like_ratio)), 4))
print("comment_rate -> shape", comment_rate.shape, "| NaN:", np.isnan(comment_rate).sum(),
      "| median:", round(float(np.nanmedian(comment_rate)), 4))

like_ratio   -> shape (46000,) | NaN: 551 | median: 0.9489
comment_rate -> shape (46000,) | NaN: 36 | median: 0.0033


### (b) Min-max normalisation
`(x - min) / (max - min)` rescales each array to the 0–1 range so views, likes and comments can be combined fairly. `np.nanmin` / `np.nanmax` ignore the `NaN` gaps, and a guard stops a divide-by-zero if an array ever had no spread.

In [118]:
def min_max_normalize(values):
    """Rescale an array to 0-1, ignoring NaN. Returns zeros if every value is identical."""
    low, high = np.nanmin(values), np.nanmax(values)
    if high == low:
        return np.zeros_like(values, dtype=float)
    return (values - low) / (high - low)


views_norm = min_max_normalize(views)
likes_norm = min_max_normalize(likes)
comments_norm = min_max_normalize(comment_count)

for name, array in [("views_norm", views_norm), ("likes_norm", likes_norm), ("comments_norm", comments_norm)]:
    print(f"{name:<14} min={np.nanmin(array):.4f}  max={np.nanmax(array):.4f}  median={np.nanmedian(array):.6f}")

views_norm     min=0.0000  max=1.0000  median=0.000880
likes_norm     min=0.0000  max=1.0000  median=0.000371
comments_norm  min=0.0000  max=1.0000  median=0.001380


> The medians are tiny. That is expected, not a bug: a few huge videos define the maximum, so most records sit close to 0 after min-max scaling. The same reason makes raw views hard to plot (see part d).

### (c) Engagement score and the reasoning behind the weights

`engagement_score = 0.2 × views_norm + 0.3 × likes_norm + 0.5 × comments_norm`

| Signal | Weight | Why |
|---|---:|---|
| **Views** | **0.2** | The weakest evidence of engagement. A view can be a single tap or an autoplay, and it costs the viewer nothing. It still matters because it shows reach, so it keeps a small weight. |
| **Likes** | **0.3** | A like needs a deliberate action, so it says more than a view. It is still a one-tap reaction. |
| **Comments** | **0.5** | Writing a comment takes the most effort, so it is the strongest sign that a video made people react. It gets the largest weight. |

The weights add up to **1.0**, so each one reads as "the share of the score that comes from this signal".

**Limitations to keep in mind**
1. Min-max scaling of raw counts is dominated by the largest videos, so most scores are very small. Use the score to **rank** records; do not read its absolute size.
2. Videos with comments disabled cannot earn comment points, so they score lower by design.
3. The weights are a judgement call, not a measured truth. Different weights would reorder the top 20.

In [119]:
assert np.isclose(W_VIEWS + W_LIKES + W_COMMENTS, 1.0), "weights must sum to 1"

engagement_score = W_VIEWS * views_norm + W_LIKES * likes_norm + W_COMMENTS * comments_norm

print("Score shape :", engagement_score.shape)
print("NaN scores  :", np.isnan(engagement_score).sum(), "(records with a blank likes or comment_count)")
print("Min / Max   :", np.nanmin(engagement_score), "/", np.nanmax(engagement_score))

Score shape : (46000,)
NaN scores  : 48 (records with a blank likes or comment_count)
Min / Max   : 9.473865373765817e-07 / 0.8845116201390049


### (d) Log-scaled views

Raw views are hard to plot because they span several orders of magnitude: the median video has a few hundred thousand views while the largest has hundreds of millions. On a normal axis the huge videos stretch the scale and squash almost every other record into one corner, so the shape of the distribution is invisible. `np.log1p(x)` = `log(1 + x)` compresses the range and stays safe if a video has 0 views.

In [120]:
print("Views  min / median / max:", np.nanmin(views), "/", np.nanmedian(views), "/", np.nanmax(views))

log_views = np.log1p(views)
print("Log views min / median / max:", log_views.min().round(2), "/", np.median(log_views).round(2), "/", log_views.max().round(2))
print("log_views shape:", log_views.shape)

Views  min / median / max: 1066 / 318389.0 / 360760164
Log views min / median / max: 6.97 / 12.67 / 19.7
log_views shape: (46000,)


### Top 10% by engagement score (boolean mask)
The 90th-percentile score is the cut-off. `engagement_score >= cutoff` is a boolean mask; records with a `NaN` score compare as `False`, so they are never flagged.

In [121]:
cutoff = np.nanpercentile(engagement_score, 100 * (1 - TOP_SHARE))
top_mask = engagement_score >= cutoff

print("Cut-off score         :", cutoff)
print("Flagged records       :", top_mask.sum(), "of", top_mask.size)
print("Top-10% views  (mean / median):", round(float(views[top_mask].mean())), "/", float(np.median(views[top_mask])))
print("All records    (mean / median):", round(float(views.mean())), "/", float(np.median(views)))

# Store the results as DataFrame columns. Row order is identical, so the arrays align one-to-one.
trending["like_ratio"] = like_ratio
trending["comment_rate"] = comment_rate
trending["log_views"] = log_views
trending["engagement_score"] = engagement_score
trending["top_10pct"] = top_mask

Cut-off score         : 0.0062429832930557955
Flagged records       : 4596 of 46000
Top-10% views  (mean / median): 7681883 / 3697753.5
All records    (mean / median): 1179665 / 318389.0


## 3. Reshape and broadcast

### Days × countries array of daily trending counts
`pivot_table` counts the records for every (day, country) pair and fills missing pairs with 0, so no Python loop is needed. Rows are days, columns are countries. The dates are parsed to real dates first because the raw text (`yy.dd.mm`) would not sort chronologically.

In [122]:
trending_day = pd.to_datetime(trending["trending_date"], format=DATE_FORMAT)

daily_counts = (
    trending.assign(day=trending_day)
    .pivot_table(index="day", columns="country", values="video_id", aggfunc="count", fill_value=0)
    .reindex(columns=COUNTRIES, fill_value=0)
)

table = daily_counts.to_numpy()            # shape: (days, countries)
print("Days x countries:", table.shape, "| dtype:", table.dtype)
print("Column totals   :", dict(zip(COUNTRIES, table.sum(axis=0).tolist())))
daily_counts.head()

Days x countries: (80, 3) | dtype: int64
Column totals   : {'US': 16000, 'GB': 15000, 'IN': 15000}


country,US,GB,IN
day,,,
2017-11-14,200,200,200
2017-11-15,200,200,200
2017-11-16,200,200,200
2017-11-17,200,200,200
2017-11-18,200,200,200


### `reshape` and `flatten`
`flatten()` returns a 1-D copy read row by row. `reshape` re-cuts the same values into a new shape; it is **not** a transpose. `.T` swaps rows and columns, which is what "one row per country" really needs.

In [123]:
n_days, n_countries = table.shape

flat = table.flatten()
print("flatten()      :", flat.shape)

reshaped = table.reshape(n_countries, n_days)
print("reshape(3, n)  :", reshaped.shape)
print("reshape equals transpose?", np.array_equal(reshaped, table.T))
print("table.T        :", table.T.shape, "<- the correct 'one row per country' layout")

flatten()      : (240,)
reshape(3, n)  : (3, 80)
reshape equals transpose? False
table.T        : (3, 80) <- the correct 'one row per country' layout


### Broadcasting
Broadcasting stretches a smaller array across a larger one so no loop is needed.

- **Share of each day's total:** `(days, 3) / (days, 1)` — the single column of daily totals is stretched across the three countries. `keepdims=True` keeps the `(days, 1)` shape that makes this work.
- **Value relative to the country's average:** `(days, 3) / (3,)` — each column is divided by its own country's average.

`np.divide(..., where=...)` protects against a zero total or a zero average.

In [124]:
# --- each country's share of each day's total ---
day_totals = table.sum(axis=1, keepdims=True)                       # (days, 1)
share = np.divide(table, day_totals, out=np.zeros(table.shape), where=(day_totals != 0))

print("day_totals shape:", day_totals.shape, "| smallest daily total:", day_totals.min())
print("share shape     :", share.shape)
print("every day sums to 1?", np.allclose(share.sum(axis=1), 1))

# --- each day's value relative to that country's average ---
country_avg = table.mean(axis=0)                                     # (3,)
relative = np.divide(table, country_avg, out=np.zeros(table.shape), where=(country_avg != 0))

print("\ncountry averages:", dict(zip(COUNTRIES, country_avg.round(2).tolist())))
print("relative shape  :", relative.shape)
print("column means (should be 1.0):", relative.mean(axis=0).round(4))

pd.DataFrame(share, index=daily_counts.index, columns=COUNTRIES).round(3).head()

day_totals shape: (80, 1) | smallest daily total: 200
share shape     : (80, 3)
every day sums to 1? True

country averages: {'US': 200.0, 'GB': 187.5, 'IN': 187.5}
relative shape  : (80, 3)
column means (should be 1.0): [1. 1. 1.]


,US,GB,IN
day,,,
2017-11-14,0.333,0.333,0.333
2017-11-15,0.333,0.333,0.333
2017-11-16,0.333,0.333,0.333
2017-11-17,0.333,0.333,0.333
2017-11-18,0.333,0.333,0.333


> The country average divides by *all* days in the window. If a country has fewer trending days than the others, its missing days count as 0 and pull its average down.

## 4. Loading and inspection

The country CSV files were loaded with `pd.read_csv()` and tagged with a `country` column in Section 0. Here each raw DataFrame is inspected with `shape`, `head()`, `info()` and `describe()`.

### 4.1 Each country DataFrame

In [125]:
for country, frame in country_frames.items():
    print("=" * 70)
    print(f"{country}  |  shape = {frame.shape}")
    print("=" * 70)
    display(frame.head())
    frame.info()
    display(frame.describe().T)

US  |  shape = (16080, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
3,WOY0nn7HCsO,17.14.11,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27T22:21:47.000Z,Festival|Guitar,8083,176.0,8,19.0,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
4,2Lc_Cs5VVr_,17.14.11,Surprising Election Recipe,UltimateVlog96,27,2017-11-09T13:20:20.000Z,Festival,373161,11305.0,167,891.0,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080 entries, 0 to 16079
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                16080 non-null  object 
 1   trending_date           16080 non-null  object 
 2   title                   16080 non-null  object 
 3   channel_title           16080 non-null  object 
 4   category_id             16080 non-null  int64  
 5   publish_time            16080 non-null  object 
 6   tags                    16080 non-null  object 
 7   views                   16080 non-null  int64  
 8   likes                   16076 non-null  float64
 9   dislikes                16080 non-null  int64  
 10  comment_count           16068 non-null  float64
 11  thumbnail_link          16080 non-null  object 
 12  comments_disabled       16080 non-null  bool   
 13  ratings_disabled        16080 non-null  bool   
 14  video_error_or_removed  16080 non-null

,count,mean,std,min,25%,50%,75%,max
category_id,16080.0,1.897587e+01,7.590729e+00,1.0,10.00,23.0,24.00,28.0
views,16080.0,1.012289e+06,3.746761e+06,1621.0,117307.75,289072.5,750719.00,129229245.0
likes,16076.0,3.950464e+04,1.719414e+05,0.0,3353.00,9229.0,26267.00,6795138.0
dislikes,16080.0,2.115772e+03,7.973928e+03,0.0,194.00,540.0,1506.00,247546.0
comment_count,16068.0,3.118738e+03,1.006018e+04,0.0,375.00,999.0,2613.25,318346.0


GB  |  shape = (15075, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,u_nKn025TZS,17.14.11,Wild Winter Prank,OfficialRecipe78,28,2017-10-18T07:42:02.000Z,[none],84344,2155.0,163,475.0,https://i.ytimg.com/vi/u_nKn025TZS/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
1,FoQAEGxMMQD,17.14.11,Wild Cricket Tutorial FINALE,FinalTrailer97,26,2017-11-14T12:56:18.000Z,Minecraft|Paris,230434,5412.0,330,1704.0,https://i.ytimg.com/vi/FoQAEGxMMQD/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
2,wHqDBuStXq3,17.14.11,Epic Space Challenge,UltimateReview54,23,2017-11-13T06:19:10.000Z,Paris,338852,16813.0,1333,1385.0,https://i.ytimg.com/vi/wHqDBuStXq3/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
3,ZId0IGzSaaZ,17.14.11,Quiet Paris Prank LIVE?!,FinalUpdate79,24,2017-11-14T11:37:33.000Z,Phones,160896,4221.0,483,1342.0,https://i.ytimg.com/vi/ZId0IGzSaaZ/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB
4,1-xA4eRrYSH,17.14.11,Nova Kane - Chess (Official Video)!,QuietInterview9,10,2017-11-14T03:13:26.000Z,Chess|Minecraft|Festival|Guitar,59441,1475.0,77,60.0,https://i.ytimg.com/vi/1-xA4eRrYSH/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,GB


... (2 outputs left out)

IN  |  shape = (15075, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,UcYiQY22yfG,17.14.11,Sunny Park - Cats (Lyric Video)!,QuietChallenge75,10,2017-11-14T05:57:11.000Z,Robots|Election,217915,14918.0,1363,662.0,https://i.ytimg.com/vi/UcYiQY22yfG/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
1,EutcTEIciae,17.14.11,Perfect Pasta Unboxing?!,UltimateUnboxing65,1,2017-11-08T13:18:04.000Z,Paris|Bangalore|Winter,115014,2739.0,287,209.0,https://i.ytimg.com/vi/EutcTEIciae/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
2,rvuZ5CsBpg8,17.14.11,Wild Budget Challenge!!,Brand NewSecret51,24,2017-11-14T18:16:59.000Z,Space,51028,1786.0,322,365.0,https://i.ytimg.com/vi/rvuZ5CsBpg8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
3,2yJGS3uxgMP,17.14.11,Quiet Chess Mystery,UltimateUnboxing18,23,2017-11-11T20:04:47.000Z,Budget|Festival|Minecraft|Guitar,143370,5224.0,305,571.0,https://i.ytimg.com/vi/2yJGS3uxgMP/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
4,W5pdmLZu7GK,17.14.11,Quiet Festival Prank,HonestInterview42,24,2017-11-14T05:23:55.000Z,[none],390220,15079.0,947,1942.0,https://i.ytimg.com/vi/W5pdmLZu7GK/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15075 entries, 0 to 15074
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                15075 non-null  object 
 1   trending_date           15075 non-null  object 
 2   title                   15075 non-null  object 
 3   channel_title           15075 non-null  object 
 4   category_id             15075 non-null  int64  
 5   publish_time            15075 non-null  object 
 6   tags                    15075 non-null  object 
 7   views                   15075 non-null  int64  
 8   likes                   15071 non-null  float64
 9   dislikes                15075 non-null  int64  
 10  comment_count           15063 non-null  float64
 11  thumbnail_link          15075 non-null  object 
 12  comments_disabled       15075 non-null  bool   
 13  ratings_disabled        15075 non-null  bool   
 14  video_error_or_removed  15075 non-null

,count,mean,std,min,25%,50%,75%,max
category_id,15075.0,19.836086,7.341271e+00,1.0,17.0,24.0,24.0,28.0
views,15075.0,954592.446567,3.035647e+06,3542.0,152165.5,355802.0,863998.0,123208090.0
likes,15071.0,33218.333289,1.042103e+05,0.0,4107.0,10915.0,29262.0,3634665.0
dislikes,15075.0,1937.757877,5.791483e+03,0.0,257.0,681.0,1815.5,287063.0
comment_count,15063.0,3210.599283,8.251638e+03,0.0,496.0,1243.0,3101.0,346103.0


### 4.2 Combined DataFrame (raw, before any cleaning)

In [126]:
print("shape =", combined_raw.shape)
display(combined_raw.head())
combined_raw.info()
display(combined_raw.describe().T)

shape = (46230, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,g_N-w8LeAi8,17.14.11,Honest Football Prank LEAKED,QuietSpeech10,22,2017-11-14T07:04:28.000Z,Monsoon|Robots,48555,2189.0,180,136.0,https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
1,Ugyql2NlPAd,17.14.11,Surprising Cricket Review,IncredibleChallenge41,25,2017-11-12T22:35:48.000Z,Winter,198385,1159.0,492,1446.0,https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
2,8fE19jkxJ9n,17.14.11,Mira Solis - Football (Live) OFFICIAL,WildTutorial42,10,2017-11-13T08:40:22.000Z,Cricket,231656,8742.0,896,432.0,https://i.ytimg.com/vi/8fE19jkxJ9n/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
3,WOY0nn7HCsO,17.14.11,Zed Rivera - Space (Lyric Video)?!,ShockingInterview30,10,2017-10-27T22:21:47.000Z,Festival|Guitar,8083,176.0,8,19.0,https://i.ytimg.com/vi/WOY0nn7HCsO/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US
4,2Lc_Cs5VVr_,17.14.11,Surprising Election Recipe,UltimateVlog96,27,2017-11-09T13:20:20.000Z,Festival,373161,11305.0,167,891.0,https://i.ytimg.com/vi/2Lc_Cs5VVr_/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,US


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46230 entries, 0 to 46229
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   video_id                46230 non-null  object 
 1   trending_date           46230 non-null  object 
 2   title                   46230 non-null  object 
 3   channel_title           46230 non-null  object 
 4   category_id             46230 non-null  int64  
 5   publish_time            46230 non-null  object 
 6   tags                    46230 non-null  object 
 7   views                   46230 non-null  int64  
 8   likes                   46218 non-null  float64
 9   dislikes                46230 non-null  int64  
 10  comment_count           46194 non-null  float64
 11  thumbnail_link          46230 non-null  object 
 12  comments_disabled       46230 non-null  bool   
 13  ratings_disabled        46230 non-null  bool   
 14  video_error_or_removed  46230 non-null

,count,mean,std,min,25%,50%,75%,max
category_id,46230.0,1.864534e+01,7.646745e+00,1.0,10.00,22.0,24.00,28.0
views,46230.0,1.178603e+06,5.916188e+06,1066.0,129463.25,318277.5,824050.50,360760164.0
likes,46218.0,4.580005e+04,2.969554e+05,0.0,3656.00,10201.0,28987.75,27537822.0
dislikes,46230.0,2.325984e+03,1.051896e+04,0.0,213.00,594.0,1666.00,618175.0
comment_count,46194.0,3.449773e+03,1.366515e+04,0.0,404.00,1074.0,2779.00,778390.0


### 4.3 Fixing data types
Two problems show up in `info()`: the dates are stored as text (`object`), and `likes` / `comment_count` are `float64` only because they contain blanks. The working table `trending` is converted:

| Column | From | To | Reason |
|---|---|---|---|
| `trending_date` | text `yy.dd.mm` | `datetime64` | needed for sorting, differences and lifespan |
| `publish_time` | ISO text ending in `Z` | timezone-aware datetime (UTC) | needed for weekday, hour and `days_to_trend` |
| `likes`, `comment_count` | `float64` | nullable `Int64` | these are whole-number counts; `Int64` keeps blanks as `<NA>` without turning counts into decimals |

The flag columns are already `bool`, `views` / `dislikes` / `category_id` are already integers, and the text columns stay as text.

In [127]:
trending["trending_date"] = pd.to_datetime(trending["trending_date"], format=DATE_FORMAT)
trending["publish_time"] = pd.to_datetime(trending["publish_time"], utc=True, errors="coerce")
trending["likes"] = trending["likes"].astype("Int64")
trending["comment_count"] = trending["comment_count"].astype("Int64")

print("trending shape =", trending.shape)
trending.info()

trending shape = (46000, 22)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   video_id                46000 non-null  object             
 1   trending_date           46000 non-null  datetime64[ns]     
 2   title                   46000 non-null  object             
 3   channel_title           46000 non-null  object             
 4   category_id             46000 non-null  int64              
 5   publish_time            46000 non-null  datetime64[ns, UTC]
 6   tags                    46000 non-null  object             
 7   views                   46000 non-null  int64              
 8   likes                   45988 non-null  Int64              
 9   dislikes                46000 non-null  int64              
 10  comment_count           45964 non-null  Int64              
 11  thumbnail_li

### 4.4 Data-quality issues and how each was handled
The table is built from the data itself, so every count is live.

In [128]:
published_day = trending["publish_time"].dt.tz_localize(None).dt.normalize()

quality_issues = pd.DataFrame(
    [
        ("Duplicate rows (same country + video_id + trending_date)",
         int(combined_raw.duplicated(subset=DUP_KEY).sum()),
         "Dropped, keeping the first copy (Section 6). Already applied to `trending`."),
        ("Missing description",
         int(trending["description"].isna().sum()),
         "Kept the rows; filled with an empty string for text work (Section 6). Not needed for the numeric analysis."),
        ("Missing likes",
         int(trending["likes"].isna().sum()),
         "Kept as missing, not replaced by 0 (Section 6). Their engagement score is NaN."),
        ("Missing comment_count",
         int(trending["comment_count"].isna().sum()),
         "Kept as missing, not replaced by 0 (Section 6). Their engagement score is NaN."),
        ("Removed / errored videos (video_error_or_removed = True)",
         int(trending["video_error_or_removed"].eq(True).sum()),
         "Excluded from the analysis table `analysis` (Section 4.5), because their statistics cannot be trusted."),
        ("Like ratio undefined (likes + dislikes = 0, or likes missing)",
         int(trending["like_ratio"].isna().sum()),
         "Ratio left as NaN by safe division (Section 2); never turned into 0."),
        ("Comments disabled",
         int(trending["comments_disabled"].eq(True).sum()),
         "Kept; these videos cannot earn comment points in the score (limitation noted in Section 2)."),
        ("Negative counts in views / likes / dislikes / comment_count",
         int((trending[NUMERIC_COLUMNS] < 0).sum().sum()),
         "None expected; checked so that no cleaning is needed."),
        ("Unparseable trending_date or publish_time",
         int(trending["trending_date"].isna().sum() + trending["publish_time"].isna().sum()),
         "Dates converted with an explicit format; any failure would show here as NaT."),
        ("Inconsistent dates: published after the trending date",
         int((published_day > trending["trending_date"]).sum()),
         "Would indicate a bad timestamp. Checked again in Section 9."),
        ("Dates stored as text in two different formats (yy.dd.mm and ISO)",
         len(trending),
         "Converted to datetimes (Section 4.3). Count = every row."),
        ("likes / comment_count stored as float",
         len(trending),
         "Converted to nullable Int64 (Section 4.3). Count = every row."),
    ],
    columns=["Data-quality issue", "Rows affected", "How it was handled"],
)

with pd.option_context("display.max_colwidth", 120):
    display(quality_issues)

,Data-quality issue,Rows affected,How it was handled
0,Duplicate rows (same country + video_id + trending_date),230,"Dropped, keeping the first copy (Section 6). Already applied to `trending`."
1,Missing description,1615,Kept the rows; filled with an empty string for text work (Section 6). Not needed for the numeric analysis.
2,Missing likes,12,"Kept as missing, not replaced by 0 (Section 6). Their engagement score is NaN."
3,Missing comment_count,36,"Kept as missing, not replaced by 0 (Section 6). Their engagement score is NaN."
4,Removed / errored videos (video_error_or_removed = True),92,"Excluded from the analysis table `analysis` (Section 4.5), because their statistics cannot be trusted."
5,"Like ratio undefined (likes + dislikes = 0, or likes missing)",551,Ratio left as NaN by safe division (Section 2); never turned into 0.
6,Comments disabled,1041,Kept; these videos cannot earn comment points in the score (limitation noted in Section 2).
7,Negative counts in views / likes / dislikes / comment_count,0,None expected; checked so that no cleaning is needed.
8,Unparseable trending_date or publish_time,0,Dates converted with an explicit format; any failure would show here as NaT.
9,Inconsistent dates: published after the trending date,0,Would indicate a bad timestamp. Checked again in Section 9.


### 4.5 Analysis table
Removed or errored videos are excluded from the analysis from here on (`EXCLUDE_REMOVED` in the setup cell controls this). Sections 1–3 used every deduplicated record; the effect of those rows on the min-max scaling is negligible because they are a tiny share of the data. Sections 5–10 all use `analysis`, so the numbers across them are consistent.

In [129]:
removed_mask = trending["video_error_or_removed"].eq(True)

analysis = (trending.loc[~removed_mask] if EXCLUDE_REMOVED else trending).copy()

print(f"trending (deduplicated)      : {len(trending):,}")
print(f"removed / errored excluded   : {int(removed_mask.sum()) if EXCLUDE_REMOVED else 0:,}")
print(f"analysis table               : {len(analysis):,}")

trending (deduplicated)      : 46,000
removed / errored excluded   : 92
analysis table               : 45,908


## 5. Filtering

`&` means AND and `|` means OR. Each condition must be wrapped in its own parentheses. `loc` selects rows *and* columns by label, `iloc` selects by position, and `sort_values` orders the result.

### (a) Records with 1 million+ views **and** more than 50,000 likes

In [130]:
big_hits = (
    analysis.loc[
        (analysis["views"] >= 1_000_000) & (analysis["likes"] > 50_000),
        ["title", "channel_title", "country", "views", "likes"],
    ]
    .sort_values("views", ascending=False)
)

print(f"{len(big_hits):,} records have 1M+ views and more than 50,000 likes. Top 10 by views:")
display(big_hits.head(10))

6,705 records have 1M+ views and more than 50,000 likes. Top 10 by views:


,title,channel_title,country,views,likes
26400,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,360760164,27537822
30800,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,343297002,9581910
26200,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,286714357,21885701
30601,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,282812154,7893691
24000,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,255362869,11413793
26000,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,236275528,18035565
30401,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,230704137,6439282
25800,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,210747294,16086924
23800,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,200444300,8959133
30201,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,181785943,5073905


### (b) Top 20 records by engagement score

In [131]:
top_20 = (
    analysis.sort_values("engagement_score", ascending=False)
    .iloc[:20]                                                    # iloc: first 20 rows by position
    [["title", "channel_title", "country", "views", "likes", "comment_count", "engagement_score"]]
)

display(top_20)

,title,channel_title,country,views,likes,comment_count,engagement_score
26400,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,360760164,27537822,598600,0.884512
24000,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,255362869,11413793,778390,0.765912
30800,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,343297002,9581910,663209,0.720718
26200,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,286714357,21885701,475737,0.702966
23800,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,200444300,8959133,610988,0.601194
30601,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,282812154,7893691,546360,0.593737
26000,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,236275528,18035565,392045,0.579299
25800,Lena Ortiz - Monsoon (Live),FinalSpeech81,GB,210747294,16086924,349687,0.516710
30401,The Hollow Pines - Phones (Official Video) WATCH,WildUnboxing39,GB,230704137,6439282,445693,0.484340
23600,Rook & Vale - Phones (Official Video) WATCH,PerfectTrailer65,GB,159635918,7135146,486597,0.478797


> The same video can appear more than once because each row is one trending day. The question asks for records, so rows are not collapsed.

### (c) Music **or** Gaming records with views above the overall median
The category IDs are looked up in `categories.csv` instead of being typed in by hand. The *overall median* is the median `views` of the whole analysis table.

In [132]:
music_id = categories.loc[categories["category_name"] == "Music", "category_id"].iloc[0]
gaming_id = categories.loc[categories["category_name"] == "Gaming", "category_id"].iloc[0]
print(f"Music = category {music_id}, Gaming = category {gaming_id}")

overall_median_views = analysis["views"].median()
print(f"Overall median views: {overall_median_views:,.0f}")

music_or_gaming = analysis.loc[
    ((analysis["category_id"] == music_id) | (analysis["category_id"] == gaming_id))
    & (analysis["views"] > overall_median_views),
    ["title", "category_id", "country", "views"],
].sort_values("views", ascending=False)

print(f"\n{len(music_or_gaming):,} Music or Gaming records are above the median. Top 10:")
display(music_or_gaming.head(10))

Music = category 10, Gaming = category 20
Overall median views: 318,570

9,106 Music or Gaming records are above the median. Top 10:


,title,category_id,country,views
26400,Lena Ortiz - Monsoon (Live),10,GB,360760164
30800,The Hollow Pines - Phones (Official Video) WATCH,10,GB,343297002
26200,Lena Ortiz - Monsoon (Live),10,GB,286714357
30601,The Hollow Pines - Phones (Official Video) WATCH,10,GB,282812154
24000,Rook & Vale - Phones (Official Video) WATCH,10,GB,255362869
26000,Lena Ortiz - Monsoon (Live),10,GB,236275528
30401,The Hollow Pines - Phones (Official Video) WATCH,10,GB,230704137
25800,Lena Ortiz - Monsoon (Live),10,GB,210747294
23800,Rook & Vale - Phones (Official Video) WATCH,10,GB,200444300
30201,The Hollow Pines - Phones (Official Video) WATCH,10,GB,181785943


## 6. Missing values and duplicates

### 6.1 Missing-value report
Measured on `trending` (all deduplicated records, before the removed videos are excluded) so that nothing is hidden.

In [133]:
missing_report = pd.DataFrame({
    "missing_count": trending.isna().sum(),
    "missing_percent": (trending.isna().mean() * 100).round(2),
})

missing_report = missing_report[missing_report["missing_count"] > 0]
# The ratio and score columns are derived from likes / comment_count, so they are not independent problems
missing_report = missing_report.drop(index=["like_ratio", "comment_rate", "engagement_score"], errors="ignore")
display(missing_report)

,missing_count,missing_percent
likes,12,0.03
comment_count,36,0.08
description,1615,3.51


### 6.2 Strategy for each affected column

| Column | Strategy | Justification |
|---|---|---|
| `likes` | **Keep as missing** (`NaN`) | A blank is unknown, not "zero likes". Replacing it with 0 would claim nobody liked the video and distort the like ratio. Pandas and NumPy functions skip the gaps, and the rows still carry valid views and dislikes. |
| `comment_count` | **Keep as missing** (`NaN`) | Same reasoning: unknown is not zero. These rows get no engagement score, so they cannot enter the top 10% by accident. |
| `description` | **`fillna("")`** | A missing description simply means the uploader wrote none. An empty string is accurate and keeps text operations (`.str.len()`, search) from failing. No text is invented. |

`dropna()` would also work, but it would throw away rows that have valid views and dislikes. The cell below shows exactly how many rows that would cost.

In [134]:
# fillna: description (applied to the analysis table)
analysis["description"] = analysis["description"].fillna("")
print("Missing descriptions left in analysis:", int(analysis["description"].isna().sum()))

# dropna: shown on a throw-away copy only, to measure the cost of dropping instead of keeping
if_dropped = analysis.dropna(subset=["likes", "comment_count"])
print(f"dropna(subset=['likes', 'comment_count']) would remove {len(analysis) - len(if_dropped):,} "
      f"of {len(analysis):,} rows -> not applied")

# isna: the remaining gaps in `analysis`
print("\nRemaining missing values in analysis:")
display(analysis.isna().sum()[lambda s: s > 0])

Missing descriptions left in analysis: 0
dropna(subset=['likes', 'comment_count']) would remove 48 of 45,908 rows -> not applied

Remaining missing values in analysis:


likes                12
comment_count        36
like_ratio          551
comment_rate         36
engagement_score     48
dtype: int64

### 6.3 Duplicates

**Definition.** A trending list shows a video at most once per country per day. So a record is a **duplicate** when it has the same **`country` + `video_id` + `trending_date`** as an earlier record. The first copy is kept. Using `video_id` alone would be wrong, because one video legitimately trends on many days and in many countries.

In [135]:
is_duplicate = combined_raw.duplicated(subset=DUP_KEY, keep="first")
print(f"Duplicate rows found and removed: {int(is_duplicate.sum()):,}")
print("By country:")
display(combined_raw.loc[is_duplicate, "country"].value_counts().rename("duplicates_removed").to_frame())

# Are the duplicates complete copies? (every column identical)
exact_copies = int(combined_raw.duplicated(keep="first").sum())
print(f"Rows that are exact copies on every column: {exact_copies:,}")

# drop_duplicates: result must match the `trending` table built in Section 0
deduplicated = combined_raw.drop_duplicates(subset=DUP_KEY, keep="first")
print(f"\nAfter drop_duplicates: {len(deduplicated):,} rows | matches `trending`: {len(deduplicated) == len(trending)}")
print("Duplicates left in trending:", int(trending.duplicated(subset=DUP_KEY).sum()))

# A few duplicated keys side by side
display(
    combined_raw[combined_raw.duplicated(subset=DUP_KEY, keep=False)]
    .sort_values(DUP_KEY)[DUP_KEY + ["views", "likes"]]
    .head(6)
)

Duplicate rows found and removed: 230
By country:


,duplicates_removed
country,
US,80
GB,75
IN,75


Rows that are exact copies on every column: 230

After drop_duplicates: 46,000 rows | matches `trending`: True
Duplicates left in trending: 0


,country,video_id,trending_date,views,likes
27388,GB,0F6i6fTiC5P,18.09.01,571066,13504.0
27389,GB,0F6i6fTiC5P,18.09.01,571066,13504.0
30038,GB,0Rn4mrFoI78,18.22.01,538396,16275.0
30039,GB,0Rn4mrFoI78,18.22.01,538396,16275.0
26169,GB,1SlitrSPOKi,18.03.01,1696163,44907.0
26170,GB,1SlitrSPOKi,18.03.01,1696163,44907.0


## 7. Joining category names

`categories.csv` maps `category_id` to `category_name`. The merge is a **left merge** on `category_id`:

- A **left** merge keeps every trending record, even if its `category_id` is missing from `categories.csv` (the name would be `NaN`). An **inner** merge would silently delete such records.
- `validate="many_to_one"` makes Pandas raise an error if `categories.csv` repeated a `category_id`, which would duplicate trending rows.

The checks below prove that no records were lost.

In [136]:
videos = analysis.merge(categories, on="category_id", how="left", validate="many_to_one")

print(f"Rows before merge : {len(analysis):,}")
print(f"Rows after merge  : {len(videos):,}")
print(f"Records lost      : {len(analysis) - len(videos):,}")
print(f"Unmatched category_id (name is NaN): {int(videos['category_name'].isna().sum()):,}")

assert len(videos) == len(analysis), "the merge changed the number of records"
display(videos[["title", "category_id", "category_name", "country", "views"]].head())

Rows before merge : 45,908
Rows after merge  : 45,908
Records lost      : 0
Unmatched category_id (name is NaN): 0


,title,category_id,category_name,country,views
0,Honest Football Prank LEAKED,22,People & Blogs,US,48555
1,Surprising Cricket Review,25,News & Politics,US,198385
2,Mira Solis - Football (Live) OFFICIAL,10,Music,US,231656
3,Zed Rivera - Space (Lyric Video)?!,10,Music,US,8083
4,Surprising Election Recipe,27,Education,US,373161


## 8. Aggregation

One reusable function reports the record count, average views, average likes and average engagement score for any grouping.

In [137]:
def summarise(df, by):
    """Record count, average views, average likes and average engagement score per group."""
    result = df.groupby(by).agg(
        record_count=("video_id", "count"),
        avg_views=("views", "mean"),
        avg_likes=("likes", "mean"),
        avg_engagement_score=("engagement_score", "mean"),
    )
    return result.round({"avg_views": 0, "avg_likes": 0, "avg_engagement_score": 6}).sort_values(
        "record_count", ascending=False
    )


print("By category:")
display(summarise(videos, "category_name"))

print("By country:")
display(summarise(videos, "country"))

print("By category and country:")
display(summarise(videos, ["category_name", "country"]).head(15))

By category:


,record_count,avg_views,avg_likes,avg_engagement_score
category_name,,,,
Entertainment,12244,901973.0,28970.0,0.003248
Music,10286,3193066.0,135069.0,0.007459
People & Blogs,3797,551729.0,19850.0,0.002499
Comedy,3251,602812.0,28663.0,0.002212
News & Politics,2420,283900.0,3115.0,0.001643
Film & Animation,2290,578835.0,17218.0,0.001823
Gaming,2055,509739.0,20763.0,0.002284
Sports,1919,374671.0,10458.0,0.001023
Howto & Style,1819,300403.0,10246.0,0.000866


By country:


,record_count,avg_views,avg_likes,avg_engagement_score
country,,,,
US,15968,1014564.0,39566.0,0.003000
GB,14970,1583959.0,65288.0,0.004192
IN,14970,953217.0,33187.0,0.002951


By category and country:


,,record_count,avg_views,avg_likes,avg_engagement_score
category_name,country,,,,
Entertainment,IN,5081,966237.0,31227.0,0.003405
Music,GB,4978,3729081.0,161459.0,0.008862
Entertainment,US,4342,881302.0,27710.0,0.003378
Music,US,3029,2870471.0,123039.0,0.006570
Entertainment,GB,2821,818040.0,26841.0,0.002763
Music,IN,2279,2451013.0,93421.0,0.005578
People & Blogs,US,1416,646617.0,23732.0,0.002648
Comedy,US,1358,585808.0,29691.0,0.002188
People & Blogs,GB,1289,438971.0,16044.0,0.002031


### Category distribution with `value_counts()` — counts and percentages

In [138]:
category_distribution = pd.DataFrame({
    "records": videos["category_name"].value_counts(),
    "percent": (videos["category_name"].value_counts(normalize=True) * 100).round(2),
})
display(category_distribution)

,records,percent
category_name,,
Entertainment,12244,26.67
Music,10286,22.41
People & Blogs,3797,8.27
Comedy,3251,7.08
News & Politics,2420,5.27
Film & Animation,2290,4.99
Gaming,2055,4.48
Sports,1919,4.18
Howto & Style,1819,3.96


### Publishing day and hour
Both are taken from `publish_time`, which is in **UTC**. The median is shown next to the mean because a handful of very popular videos can pull a group's average far above what a typical video in that group gets.

In [139]:
videos["publish_day"] = videos["publish_time"].dt.day_name()
videos["publish_hour"] = videos["publish_time"].dt.hour


def views_by(df, column):
    return (
        df.groupby(column)
        .agg(record_count=("video_id", "count"),
             avg_views=("views", "mean"),
             median_views=("views", "median"))
        .round(0)
        .sort_values("avg_views", ascending=False)
    )


day_stats = views_by(videos, "publish_day")
hour_stats = views_by(videos, "publish_hour")

print("By day of week (highest average views first):")
display(day_stats)

print("By hour of day (UTC), top 5:")
display(hour_stats.head(5))

print(f"Highest average views -> day: {day_stats.index[0]} | hour: {hour_stats.index[0]}:00 UTC")

By day of week (highest average views first):


,record_count,avg_views,median_views
publish_day,,,
Monday,6695,1592956.0,341225.0
Friday,6256,1285997.0,309886.0
Tuesday,7087,1257605.0,320822.0
Thursday,6585,1111215.0,310368.0
Wednesday,6534,1103037.0,323318.0
Saturday,6475,973686.0,319033.0
Sunday,6276,913034.0,305918.0


By hour of day (UTC), top 5:


,record_count,avg_views,median_views
publish_hour,,,
12,1854,3251512.0,338600.0
8,1849,2082956.0,323797.0
16,2043,1519497.0,335803.0
21,1843,1432901.0,335573.0
15,1971,1352450.0,296535.0


Highest average views -> day: Monday | hour: 12:00 UTC


> These are patterns in the trending data, not proof that publishing at a particular time *causes* more views. Compare the mean with the median and the record counts before reading too much into a single day or hour.

## 9. Time to trend and trending lifespan

### 9.1 `days_to_trend`
The number of **calendar days** between the publish date and the trending date. Both are reduced to whole dates first, so the time of day does not matter.

In [140]:
publish_date = videos["publish_time"].dt.tz_localize(None).dt.normalize()
videos["days_to_trend"] = (videos["trending_date"] - publish_date).dt.days

display(videos["days_to_trend"].describe().to_frame().T)
print("Percentiles 50 / 90 / 99:", videos["days_to_trend"].quantile([0.5, 0.9, 0.99]).tolist())
print("Trended on the day it was published (0 days):", int((videos["days_to_trend"] == 0).sum()))
print("Negative values (published after trending)  :", int((videos["days_to_trend"] < 0).sum()))

,count,mean,std,min,25%,50%,75%,max
days_to_trend,45908.0,5.090943,6.243572,0.0,1.0,3.0,6.0,47.0


Percentiles 50 / 90 / 99: [3.0, 12.0, 30.0]
Trended on the day it was published (0 days): 6120
Negative values (published after trending)  : 0


### 9.2 Trending lifespan
**Lifespan** = the number of *distinct days* a video stayed on the trending list in one country. `groupby` + `nunique` counts each day once.

In [141]:
video_lifespan = (
    videos.groupby(["country", "video_id"])
    .agg(
        title=("title", "first"),
        channel_title=("channel_title", "first"),
        category_name=("category_name", "first"),
        trending_lifespan=("trending_date", "nunique"),
    )
    .reset_index()
)

# A video should keep one category; confirm so that 'first' is safe
print("Max categories per video+country:", videos.groupby(["country", "video_id"])["category_name"].nunique().max())
print(f"Video-country pairs: {len(video_lifespan):,}")

print("\nLifespan distribution (days on the list):")
display(video_lifespan["trending_lifespan"].describe().to_frame().T)

Max categories per video+country: 1
Video-country pairs: 15,047

Lifespan distribution (days on the list):


,count,mean,std,min,25%,50%,75%,max
trending_lifespan,15047.0,3.050974,2.916785,1.0,1.0,2.0,4.0,25.0


### 10 longest-lasting videos
Ties are common at the top, so equal lifespans are ordered by `video_id` to keep the list stable between runs.

In [142]:
longest_lasting = (
    video_lifespan.sort_values(["trending_lifespan", "video_id"], ascending=[False, True])
    .head(10)
    [["video_id", "country", "title", "channel_title", "category_name", "trending_lifespan"]]
    .reset_index(drop=True)
)
display(longest_lasting)

,video_id,country,title,channel_title,category_name,trending_lifespan
0,BsTMdCTYs7P,US,Surprising Guitar Update,Brand NewUnboxing21,Entertainment,25
1,DYLzrqG-SrK,GB,Official Football Update,InsaneReaction46,Entertainment,25
2,EZaG9_2Yi1u,GB,KAIRO - FESTIVAL (LIVE),QuietChallenge75,Music,25
3,Hh9b0BBsne0,GB,Kairo - Phones (Lyric Video)!,PerfectTrailer65,Music,25
4,Nc1MUQvmyWO,US,Sunny Park - Cats (Official Video),ShockingInterview30,Music,25
5,NqokCmZykzK,GB,Anya Lux - Paris (Lyric Video),QuietInterview9,Music,25
6,O6zdCRwWdf1,GB,MIRA SOLIS - WINTER (LYRIC VIDEO)?,FinalTrailer17,Music,25
7,RdqSDDng84n,GB,Lena Ortiz - Monsoon (Live),FinalSpeech81,Music,25
8,Z1aFIGyR5zv,GB,Nova Kane - Robots (Official Video),QuietChallenge75,Music,25
9,cz0dTlZWwSk,GB,Final Cars Speech WATCH,UltimateUnboxing65,Film & Animation,25


### Average lifespan per category

In [143]:
average_lifespan_by_category = (
    video_lifespan.groupby("category_name")
    .agg(average_lifespan=("trending_lifespan", "mean"), videos=("video_id", "count"))
    .round(2)
    .sort_values("average_lifespan", ascending=False)
)
display(average_lifespan_by_category)

,average_lifespan,videos
category_name,,
Music,5.32,1932
Entertainment,3.53,3465
Comedy,2.79,1166
People & Blogs,2.78,1364
Film & Animation,2.70,847
Gaming,2.64,778
Education,2.44,673
Howto & Style,2.40,758
Autos & Vehicles,2.37,425


## 10. Combining data

### 10.1 `concat()` — stacking the per-country DataFrames
The three country DataFrames were stacked into one with `pd.concat()` in Section 0. The cell below confirms that no rows were lost or added in the process.

In [144]:
per_country_rows = {country: len(frame) for country, frame in country_frames.items()}
print("Rows per country:", per_country_rows)
print("Sum of countries:", sum(per_country_rows.values()), "| combined_raw rows:", len(combined_raw))
assert sum(per_country_rows.values()) == len(combined_raw)

print("\nRows per country in the stacked table:")
display(combined_raw["country"].value_counts().rename("rows").to_frame())

Rows per country: {'US': 16080, 'GB': 15075, 'IN': 15075}
Sum of countries: 46230 | combined_raw rows: 46230

Rows per country in the stacked table:


,rows
country,
US,16080
GB,15075
IN,15075


### 10.2 Per-video summary
One row per video (and channel). `peak_views` is the highest view count the video reached. The `views` column is cumulative, so adding every daily row together would count the same views many times.

In [145]:
per_video_summary = (
    videos.groupby(["video_id", "channel_title"], as_index=False)
    .agg(peak_views=("views", "max"), countries_trended=("country", "nunique"))
)

print("per_video_summary shape:", per_video_summary.shape)
display(per_video_summary.head())

per_video_summary shape: (14688, 4)


,video_id,channel_title,peak_views,countries_trended
0,--UQXlrk5E0,UltimateUnboxing65,179541,1
1,--j83Wp5WrQ,AmazingSpeech11,68090,1
2,--pb5oKkYrH,QuietChallenge75,582515,1
3,-0B4eg2Srpi,FinalBreakdown11,396797,1
4,-0dHuLKRPYS,OfficialUnboxing6,198202,1


### 10.3 Per-channel summary

In [146]:
per_channel_summary = (
    per_video_summary.groupby("channel_title", as_index=False)
    .agg(trending_videos=("video_id", "nunique"), total_views=("peak_views", "sum"))
)

print("per_channel_summary shape:", per_channel_summary.shape)
display(per_channel_summary.head())

per_channel_summary shape: (438, 3)


,channel_title,trending_videos,total_views
0,AmazingBreakdown51,4,612225
1,AmazingChallenge30,3,358277
2,AmazingChallenge43,7,2344012
3,AmazingChallenge55,43,27675306
4,AmazingChallenge7,2,445985


### 10.4 `merge()` — per-video table with per-channel table
Each video row receives its channel's totals, so a video can be seen in the context of its channel. The channel columns are renamed *before* the merge so the result has no confusing `_x` / `_y` suffixes. `validate="many_to_one"` confirms many videos map to one channel row.

In [147]:
channel_columns = per_channel_summary.rename(
    columns={"trending_videos": "channel_trending_videos", "total_views": "channel_total_views"}
)

video_with_channel = per_video_summary.merge(
    channel_columns, on="channel_title", how="left", validate="many_to_one"
)
video_with_channel["share_of_channel_views"] = (
    video_with_channel["peak_views"] / video_with_channel["channel_total_views"]
).round(4)

print(f"Rows before: {len(per_video_summary):,} | after merge: {len(video_with_channel):,}")
assert len(video_with_channel) == len(per_video_summary)
display(video_with_channel.head())

Rows before: 14,688 | after merge: 14,688


,video_id,channel_title,peak_views,countries_trended,channel_trending_videos,channel_total_views,share_of_channel_views
0,--UQXlrk5E0,UltimateUnboxing65,179541,1,522,297427242,0.0006
1,--j83Wp5WrQ,AmazingSpeech11,68090,1,154,117836877,0.0006
2,--pb5oKkYrH,QuietChallenge75,582515,1,466,1333310585,0.0004
3,-0B4eg2Srpi,FinalBreakdown11,396797,1,164,143515053,0.0028
4,-0dHuLKRPYS,OfficialUnboxing6,198202,1,28,11377223,0.0174


### Result: each channel's number of trending videos and total views (top 10 by total views)

In [148]:
channel_ranking = per_channel_summary.sort_values("total_views", ascending=False).reset_index(drop=True)
display(channel_ranking.head(10))

,channel_title,trending_videos,total_views
0,QuietChallenge75,466,1333310585
1,WildUnboxing39,56,582453017
2,FinalTrailer17,161,492211013
3,PerfectTutorial78,98,477526034
4,PerfectTrailer65,66,453035336
5,FinalSpeech81,34,442854302
6,SurprisingHighlights41,123,401045203
7,OfficialReview41,457,392673620
8,QuietInterview9,88,342109758
9,Brand NewChallenge45,302,305611036


## Summary

- **NumPy (Sections 1–3):** four arrays with `shape` and `dtype` reported; safe ratios, min-max normalisation, a weighted engagement score, log-scaled views, a top-10% mask, and a days × countries table with reshape and broadcasting. None of these steps uses a loop.
- **Pandas (Sections 4–7):** raw per-country and combined inspection, type fixes, a live data-quality table, filtering with `&`, `|`, `loc`, `iloc` and sorting, missing-value and duplicate decisions, and a left merge on category with a no-records-lost check.
- **Analysis (Sections 8–10):** `groupby`/`agg` by category and country, `value_counts` distribution, best publishing day and hour, `days_to_trend`, trending lifespan, and per-channel totals through `concat` and `merge`.

**Limitations**
- The engagement score uses judgement-based weights and raw min-max scaling, so it is best used for ranking.
- Each row is a video on one trending day, so popular videos appear many times. Averages over rows give those videos extra weight.
- Publishing day and hour patterns describe videos that *made* the trending list; they do not show that timing causes success.